In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
# define parameters and directory
subj = 'sub-LTP349'
task = 'task-ltpFR2'
sess = 'ses-0'
main_dir = '/Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/'
raw_dir = os.path.join(main_dir, 'raw', subj)
data_dir = os.path.join(main_dir, 'data', subj)
print(raw_dir)
print(data_dir)

/Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/raw/sub-LTP349
/Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/data/sub-LTP349


In [4]:
# read event log file
fname = os.path.join(raw_dir, '%s', 'eeg', '%s_%s_%s_events.tsv')%(sess, subj, sess, task)
df = pd.read_csv(fname, sep='\t')

# loop over blocks
block_arr = df.trial.unique()[1:]
beh_df_enc_all = []
beh_df_rec_all = []

for block_num in block_arr:

    # encode dataframe
    enc_df = df[(df.trial_type == 'WORD') & (df.trial == block_num)]

    # retrieval dataframe
    rec_df = df[((df.trial_type == 'REC_START') | (df.trial_type == 'REC_WORD')) & (df.trial == block_num)]

    # trial time info
    n_item = rec_df.shape[0]
    trial_sample = rec_df['sample'][0:n_item-1].values
    speech_sample = rec_df['sample'][1:n_item].values
    trial_onset = rec_df['onset'][0:n_item-1].values*1000
    speech_onset = rec_df['onset'][1:n_item].values*1000
    resp_time = speech_onset - trial_onset

    # behavioral dataframe
    beh_df_rec = rec_df.iloc[1:,[5,2,8,9]]
    beh_df_rec.rename(columns={"trial": "block"}, inplace=True)

    # recalled performance
    n_item = beh_df_rec.shape[0]
    pos_list = []
    acc_list = []
    for i in range(n_item):
    
        enc_pos = np.where(enc_df.item_name == beh_df_rec.iloc[i].item_name)[0]
        if len(enc_pos) > 0:
            pos_list.append(enc_pos[0])
            acc_list.append(1)
        else:
            pos_list.append(np.nan)
            acc_list.append(0)

    # organize dataframe for retrieval
    beh_df_rec['trial_sample'] = trial_sample
    beh_df_rec['speech_sample'] = speech_sample
    beh_df_rec['trial_onset'] = trial_onset
    beh_df_rec['speech_onset'] = speech_onset
    beh_df_rec['resp_time'] = resp_time
    beh_df_rec['resp_acc'] = acc_list
    beh_df_rec['enc_pos'] = pos_list

    # organize dataframe for encoding
    beh_df_enc = enc_df.iloc[:,[5,2,8,9,10,3,0]]

    # append dataframe
    beh_df_enc_all.append(beh_df_enc)
    beh_df_rec_all.append(beh_df_rec)

# combine across blocks
beh_df_enc_all = pd.concat(beh_df_enc_all, ignore_index=False)
beh_df_rec_all = pd.concat(beh_df_rec_all, ignore_index=False)
print('%s: %s items' %(sess, beh_df_enc_all.shape[0]) )
print('%s: %s items' %(sess, beh_df_rec_all.shape[0]) )

# save data
fname = os.path.join(data_dir, '%s', 'beh-enc_%s_%s_%s.csv')%(sess, subj, sess, task)
beh_df_enc_all.to_csv(fname, index=False, na_rep='NaN')
fname = os.path.join(data_dir, '%s', 'beh-rec_%s_%s_%s.csv')%(sess, subj, sess, task)
beh_df_rec_all.to_csv(fname, index=False, na_rep='NaN')
print(beh_df_rec_all.resp_acc.sum())
    

ses-0: 576 items
ses-0: 272 items
257
